In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.data_ingestion import ingest_data

df = ingest_data()

print("Shape:", df.shape)
df.head()

Extracting: multilingual-customer-support-tickets.zip
Loading: aa_dataset-tickets-multi-lang-5-2-50-version.csv

Data ingestion completed.
File: aa_dataset-tickets-multi-lang-5-2-50-version.csv
Shape: (28587, 16)
Shape: (28587, 16)


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN


In [2]:
df_en = df[df["language"] == "en"].copy()

df_en["subject"] = df_en["subject"].fillna("")
df_en["body"] = df_en["body"].fillna("")

df_en["text"] = (
    df_en["subject"] + " " + df_en["body"]
).str.strip()

df_en = df_en[df_en["text"] != ""].copy()

print("English tickets:", len(df_en))
print("\nQueue distribution:")
print(df_en["queue"].value_counts())

English tickets: 16338

Queue distribution:
queue
Technical Support                  4737
Product Support                    3073
Customer Service                   2410
IT Support                         1942
Billing and Payments               1595
Returns and Exchanges               820
Service Outages and Maintenance     664
Sales and Pre-Sales                 513
Human Resources                     348
General Inquiry                     236
Name: count, dtype: int64


In [3]:
X = df_en["text"]
y = df_en["queue"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nExample text:")
print(X.iloc[0])

print("\nTarget:")
print(y.iloc[0])

X shape: (16338,)
y shape: (16338,)

Example text:
Account Disruption Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?

Target:
Technical Support


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 13070
Testing samples: 3268


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 3),
    min_df=2,
    max_features=30000,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (13070, 30000)
Testing TF-IDF shape: (3268, 30000)


In [6]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(X_train_tfidf, y_train)

y_pred_logistic = logistic_model.predict(X_test_tfidf)

print("Logistic Regression training completed.")

Logistic Regression training completed.


In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

logistic_accuracy = accuracy_score(y_test, y_pred_logistic)

logistic_precision = precision_score(
    y_test,
    y_pred_logistic,
    average="macro",
    zero_division=0
)

logistic_recall = recall_score(
    y_test,
    y_pred_logistic,
    average="macro",
    zero_division=0
)

logistic_f1 = f1_score(
    y_test,
    y_pred_logistic,
    average="macro",
    zero_division=0
)

print("Logistic Regression")
print("-------------------")
print("Accuracy :", round(logistic_accuracy, 4))
print("Precision:", round(logistic_precision, 4))
print("Recall   :", round(logistic_recall, 4))
print("Macro F1 :", round(logistic_f1, 4))

Logistic Regression
-------------------
Accuracy : 0.5379
Precision: 0.5325
Recall   : 0.6024
Macro F1 : 0.5584


In [8]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()

nb_model.fit(X_train_tfidf, y_train)

y_pred_nb = nb_model.predict(X_test_tfidf)

print("Naive Bayes training completed.")

Naive Bayes training completed.


In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

nb_accuracy = accuracy_score(y_test, y_pred_nb)

nb_precision = precision_score(
    y_test,
    y_pred_nb,
    average="macro",
    zero_division=0
)

nb_recall = recall_score(
    y_test,
    y_pred_nb,
    average="macro",
    zero_division=0
)

nb_f1 = f1_score(
    y_test,
    y_pred_nb,
    average="macro",
    zero_division=0
)

print("Naive Bayes")
print("-----------")
print("Accuracy :", round(nb_accuracy, 4))
print("Precision:", round(nb_precision, 4))
print("Recall   :", round(nb_recall, 4))
print("Macro F1 :", round(nb_f1, 4))

Naive Bayes
-----------
Accuracy : 0.41
Precision: 0.4833
Recall   : 0.2108
Macro F1 : 0.203


In [10]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(
    class_weight="balanced",
    random_state=42
)

svm_model.fit(X_train_tfidf, y_train)

y_pred_svm = svm_model.predict(X_test_tfidf)

print("Linear SVM training completed.")

Linear SVM training completed.


In [11]:
svm_accuracy = accuracy_score(y_test, y_pred_svm)

svm_precision = precision_score(
    y_test,
    y_pred_svm,
    average="macro",
    zero_division=0
)

svm_recall = recall_score(
    y_test,
    y_pred_svm,
    average="macro",
    zero_division=0
)

svm_f1 = f1_score(
    y_test,
    y_pred_svm,
    average="macro",
    zero_division=0
)

print("Linear SVM")
print("----------")
print("Accuracy :", round(svm_accuracy, 4))
print("Precision:", round(svm_precision, 4))
print("Recall   :", round(svm_recall, 4))
print("Macro F1 :", round(svm_f1, 4))

Linear SVM
----------
Accuracy : 0.668
Precision: 0.6878
Recall   : 0.6875
Macro F1 : 0.6861


In [12]:
print("MODEL COMPARISON")
print("=" * 50)

print(
    f"Logistic Regression | "
    f"Accuracy: {logistic_accuracy:.4f} | "
    f"Macro F1: {logistic_f1:.4f}"
)

print(
    f"Naive Bayes         | "
    f"Accuracy: {nb_accuracy:.4f} | "
    f"Macro F1: {nb_f1:.4f}"
)

print(
    f"Linear SVM          | "
    f"Accuracy: {svm_accuracy:.4f} | "
    f"Macro F1: {svm_f1:.4f}"
)

MODEL COMPARISON
Logistic Regression | Accuracy: 0.5379 | Macro F1: 0.5584
Naive Bayes         | Accuracy: 0.4100 | Macro F1: 0.2030
Linear SVM          | Accuracy: 0.6680 | Macro F1: 0.6861


In [13]:
from pathlib import Path
import joblib

PROJECT_ROOT = Path.cwd().parent
MODELS_DIR = PROJECT_ROOT / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    tfidf,
    MODELS_DIR / "queue_tfidf_vectorizer.pkl"
)

joblib.dump(
    svm_model,
    MODELS_DIR / "queue_classifier.pkl"
)

print("Queue model saved successfully.")

Queue model saved successfully.


In [14]:
print("\nSaved files:")

for file in sorted(MODELS_DIR.glob("*.pkl")):
    print(file.name)


Saved files:
priority_classifier.pkl
priority_scaler.pkl
priority_vectorizer.pkl
queue_classifier.pkl
queue_tfidf_vectorizer.pkl
tfidf_vectorizer.pkl


In [15]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred_svm,
    zero_division=0
))

                                 precision    recall  f1-score   support

           Billing and Payments       0.83      0.86      0.85       319
               Customer Service       0.62      0.63      0.63       482
                General Inquiry       0.79      0.66      0.72        47
                Human Resources       0.76      0.71      0.74        70
                     IT Support       0.57      0.60      0.59       388
                Product Support       0.64      0.59      0.61       615
          Returns and Exchanges       0.70      0.68      0.69       164
            Sales and Pre-Sales       0.58      0.66      0.62       103
Service Outages and Maintenance       0.70      0.80      0.74       133
              Technical Support       0.68      0.68      0.68       947

                       accuracy                           0.67      3268
                      macro avg       0.69      0.69      0.69      3268
                   weighted avg       0.67      0

In [16]:
error_df = pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_pred_svm
})

confusion_pairs = (
    error_df[error_df["actual"] != error_df["predicted"]]
    .groupby(["actual", "predicted"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

confusion_pairs.head(15)

,actual,predicted,count
43,Product Support,Technical Support,110
67,Technical Support,IT Support,88
68,Technical Support,Product Support,86
34,IT Support,Technical Support,77
64,Technical Support,Customer Service,64
15,Customer Service,Technical Support,63
36,Product Support,Customer Service,51
11,Customer Service,Product Support,41
39,Product Support,IT Support,40
30,IT Support,Product Support,32
